In [66]:
fact_orders = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
)

dim_restaurants = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/dim_restaurants"
)

dim_date = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/dim_date"
)

In [67]:
from pyspark.sql.functions import *

gold_revenue_monthly = (
    fact_orders
    .join(dim_date, "order_date")
    .groupBy("year", "month", "month_name")
    .agg(
        sum("sales_amount").alias("total_revenue"),
        sum("sales_qty").alias("total_quantity"),
        count("order_id").alias("total_orders")
    )
    .orderBy("year", "month")
)

In [68]:
display(gold_revenue_monthly)

In [69]:
gold_revenue_city = (
    fact_orders
    .join(
        dim_restaurants,
        fact_orders.restaurant_id == dim_restaurants.restaurant_id
    )
    .groupBy("city")
    .agg(
        sum("sales_amount").alias("revenue"),
        count("order_id").alias("orders")
    )
    .orderBy(desc("revenue"))
)

In [70]:
display(gold_revenue_city)

In [71]:
gold_revenue_cuisine = (
    fact_orders
    .join(
        dim_restaurants,
        fact_orders.restaurant_id == dim_restaurants.restaurant_id
    )
    .groupBy("cuisine")
    .agg(
        sum("sales_amount").alias("revenue"),
        count("order_id").alias("orders")
    )
    .orderBy(desc("revenue"))
)

In [72]:
display(gold_revenue_cuisine)

In [73]:
gold_top_restaurants = (
    fact_orders
    .join(
        dim_restaurants,
        fact_orders.restaurant_id == dim_restaurants.restaurant_id
    )
    .groupBy(
        dim_restaurants.restaurant_id,
        "name",
        "city"
    )
    .agg(
        sum("sales_amount").alias("revenue"),
        count("order_id").alias("orders")
    )
    .orderBy(desc("revenue"))
)

In [74]:
display(gold_top_restaurants)

In [75]:
fact_orders.agg(
    (
        sum("sales_amount") /
        count("order_id")
    ).alias("avg_order_value")
).show()

In [76]:
gold_revenue_monthly.write.mode("overwrite").parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/revenue_monthly"
)

gold_revenue_city.write.mode("overwrite").parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/revenue_city"
)

gold_revenue_cuisine.write.mode("overwrite").parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/revenue_cuisine"
)

gold_top_restaurants.write.mode("overwrite").parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/top_restaurants"
)